# NSW Electricity Forecasting GenAI Analyst — Enhanced

This Colab-ready notebook grounds every numerical answer in the loaded prediction data. The model may use only six approved Pandas tools: observation lookup, period analysis, charting, largest-error analysis, model comparison and demand-pattern analysis.

For public demonstration, leave `USE_GOOGLE_DRIVE = False`. To use the full private prediction file, set it to `True` and update `DRIVE_PARQUET_PATH`.

## 1. Install dependencies

In [1]:
%pip install -q "gradio>=6,<7" "openai>=3,<4" pandas numpy pyarrow plotly requests

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 23.6 MB/s eta 0:00:00


## 2. Load and validate prediction data

In [2]:
from pathlib import Path
import io
import json
import os
import sys

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import requests

USE_GOOGLE_DRIVE = False
DRIVE_PARQUET_PATH = Path("/content/drive/MyDrive/common_test_predictions.parquet")
GITHUB_DEMO_URL = (
    "https://raw.githubusercontent.com/jagminderunsw/"
    "Nsw-Electricity-Demand-Forecasting/main/results/demo_predictions.parquet"
)
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

COLUMN_RENAME = {
    "DATETIME": "datetime",
    "Actual_Demand": "actual_mw",
    "Baseline_Prediction": "baseline_mw",
    "XGBoost_Prediction": "xgboost_mw",
    "LSTM_Prediction": "lstm_mw",
    "SelectedBaselineName": "selected_baseline",
}
REQUIRED_COLUMNS = [
    "datetime", "actual_mw", "baseline_mw", "xgboost_mw", "lstm_mw"
]
MODEL_COLUMNS = {
    "Persistence baseline": "baseline_mw",
    "XGBoost": "xgboost_mw",
    "LSTM": "lstm_mw",
}

def normalise_predictions(frame):
    frame = frame.rename(columns=COLUMN_RENAME).copy()
    missing = sorted(set(REQUIRED_COLUMNS).difference(frame.columns))
    if missing:
        raise ValueError(f"Prediction data is missing columns: {missing}")
    frame["datetime"] = pd.to_datetime(frame["datetime"], errors="raise")
    frame = frame.sort_values("datetime").reset_index(drop=True)
    if frame["datetime"].duplicated().any():
        raise ValueError("Prediction data contains duplicate timestamps.")
    if frame[REQUIRED_COLUMNS].isna().any().any():
        raise ValueError("Prediction data contains missing evaluation values.")
    return frame

def load_predictions():
    candidates = []
    if USE_GOOGLE_DRIVE:
        candidates.append(DRIVE_PARQUET_PATH)
        drive_root = Path("/content/drive/MyDrive")
        if drive_root.is_dir():
            candidates.extend(sorted(drive_root.rglob("common_test_predictions.parquet")))
    candidates.extend([
        Path("results/demo_predictions.parquet"),
        Path("/content/demo_predictions.parquet"),
    ])
    for candidate in dict.fromkeys(candidates):
        if candidate.is_file():
            return normalise_predictions(pd.read_parquet(candidate)), str(candidate)
    response = requests.get(GITHUB_DEMO_URL, timeout=60)
    response.raise_for_status()
    return normalise_predictions(pd.read_parquet(io.BytesIO(response.content))), GITHUB_DEMO_URL

predictions, DATA_SOURCE = load_predictions()
print(f"Loaded {len(predictions):,} observations from {DATA_SOURCE}")
print(
    f"Period: {predictions['datetime'].min():%d %b %Y %H:%M} to "
    f"{predictions['datetime'].max():%d %b %Y %H:%M}"
)
predictions.head()

Loaded 4,320 observations from https://raw.githubusercontent.com/jagminderunsw/Nsw-Electricity-Demand-Forecasting/main/results/demo_predictions.parquet
Period: 18 Dec 2020 00:30 to 18 Mar 2021 00:00


,datetime,actual_mw,baseline_mw,xgboost_mw,lstm_mw,selected_baseline
0,2020-12-18 00:30:00,7653.540039,7989.600098,7748.378906,7729.694336,Persistence Baseline
1,2020-12-18 01:00:00,7408.129883,7653.540039,7433.844238,7390.992676,Persistence Baseline
2,2020-12-18 01:30:00,7085.470215,7408.129883,7154.297852,7086.625488,Persistence Baseline
3,2020-12-18 02:00:00,6876.750000,7085.470215,6820.886230,6833.491699,Persistence Baseline
4,2020-12-18 02:30:00,6669.049805,6876.750000,6666.026367,6694.066406,Persistence Baseline


## 3. Approved analytical tools

In [3]:
def json_value(value, digits=3):
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, (np.integer, int)):
        return int(value)
    if isinstance(value, (np.floating, float)):
        return round(float(value), digits)
    return value

def records_for_json(frame, limit=20):
    records = []
    for record in frame.head(limit).to_dict(orient="records"):
        records.append({key: json_value(value) for key, value in record.items()})
    return records

def filter_period(start_date=None, end_date=None, maximum_days=None):
    start = predictions["datetime"].min() if start_date is None else pd.Timestamp(start_date)
    end = predictions["datetime"].max() if end_date is None else pd.Timestamp(end_date)
    if end_date is not None and len(str(end_date)) <= 10:
        end = end + pd.Timedelta(days=1) - pd.Timedelta(nanoseconds=1)
    if end < start:
        raise ValueError("end_date must be on or after start_date.")
    if maximum_days is not None and end - start > pd.Timedelta(days=maximum_days):
        raise ValueError(f"Requested period is longer than {maximum_days} days.")
    frame = predictions.loc[
        predictions["datetime"].between(start, end, inclusive="both")
    ].copy()
    if frame.empty:
        raise ValueError("No observations are available for that period.")
    return frame

def metrics_for(frame, models=None):
    models = models or list(MODEL_COLUMNS)
    invalid = sorted(set(models).difference(MODEL_COLUMNS))
    if invalid:
        raise ValueError(f"Unknown model name(s): {invalid}")
    actual = frame["actual_mw"].to_numpy(dtype="float64")
    rows = []
    for model in models:
        predicted = frame[MODEL_COLUMNS[model]].to_numpy(dtype="float64")
        error = predicted - actual
        absolute = np.abs(error)
        nonzero = np.abs(actual) > 1e-9
        total_variation = np.sum((actual - actual.mean()) ** 2)
        rows.append({
            "model": model,
            "observations": len(frame),
            "mae_mw": absolute.mean(),
            "rmse_mw": np.sqrt(np.mean(error ** 2)),
            "mape_pct": np.mean(absolute[nonzero] / np.abs(actual[nonzero])) * 100,
            "r2": 1 - np.sum(error ** 2) / total_variation if total_variation else np.nan,
        })
    return pd.DataFrame(rows).sort_values("rmse_mw").reset_index(drop=True)

def get_observation(timestamp):
    target = pd.Timestamp(timestamp)
    exact = predictions.loc[predictions["datetime"] == target]
    if exact.empty:
        nearest_index = (predictions["datetime"] - target).abs().idxmin()
        row = predictions.loc[nearest_index]
        match_type = "nearest available observation"
    else:
        row = exact.iloc[0]
        match_type = "exact"
    result = {
        "match_type": match_type,
        "datetime": row["datetime"],
        "actual_mw": row["actual_mw"],
    }
    for model, column in MODEL_COLUMNS.items():
        result[f"{model}_prediction_mw"] = row[column]
        result[f"{model}_absolute_error_mw"] = abs(row[column] - row["actual_mw"])
    return {key: json_value(value) for key, value in result.items()}

def analyse_period(start_date, end_date):
    frame = filter_period(start_date, end_date)
    table = metrics_for(frame)
    peak = frame.loc[frame["actual_mw"].idxmax()]
    trough = frame.loc[frame["actual_mw"].idxmin()]
    return {
        "start": frame["datetime"].min().isoformat(),
        "end": frame["datetime"].max().isoformat(),
        "observations": len(frame),
        "actual_mean_mw": json_value(frame["actual_mw"].mean()),
        "actual_peak": {"datetime": peak["datetime"].isoformat(), "mw": json_value(peak["actual_mw"])},
        "actual_trough": {"datetime": trough["datetime"].isoformat(), "mw": json_value(trough["actual_mw"])},
        "model_metrics": records_for_json(table),
        "best_model_by_rmse": table.iloc[0]["model"],
    }

def plot_period(start_date=None, end_date=None, days=5, seed=42):
    days = int(np.clip(days, 1, 31))
    if start_date is None and end_date is None:
        first_day = predictions["datetime"].min().normalize()
        last_start = predictions["datetime"].max().normalize() - pd.Timedelta(days=days - 1)
        candidates = pd.date_range(first_day, last_start, freq="D")
        if len(candidates) == 0:
            raise ValueError("The loaded data is shorter than the requested chart window.")
        rng = np.random.default_rng(int(seed))
        start = pd.Timestamp(candidates[int(rng.integers(0, len(candidates)))])
        end = start + pd.Timedelta(days=days) - pd.Timedelta(nanoseconds=1)
        frame = filter_period(start.isoformat(), end.isoformat(), maximum_days=31)
        selection = f"reproducible random {days}-day window (seed {seed})"
    else:
        frame = filter_period(start_date, end_date, maximum_days=31)
        selection = "requested date range"
    figure = go.Figure()
    colours = {
        "Actual demand": "#111111",
        "Persistence baseline": "#7F7F7F",
        "XGBoost": "#E69F00",
        "LSTM": "#0072B2",
    }
    figure.add_trace(go.Scatter(
        x=frame["datetime"], y=frame["actual_mw"], name="Actual demand",
        line={"color": colours["Actual demand"], "width": 2.4},
    ))
    for model, column in MODEL_COLUMNS.items():
        figure.add_trace(go.Scatter(
            x=frame["datetime"], y=frame[column], name=model,
            line={"color": colours[model], "width": 1.4},
        ))
    figure.update_layout(
        title="Actual demand versus all model outputs",
        xaxis_title="Datetime", yaxis_title="Demand (MW)",
        hovermode="x unified", template="plotly_white", height=520,
        legend={"orientation": "h", "y": 1.12},
    )
    summary = analyse_period(
        frame["datetime"].min().isoformat(), frame["datetime"].max().isoformat()
    )
    return {
        "selection": selection,
        "chart_start": frame["datetime"].min().isoformat(),
        "chart_end": frame["datetime"].max().isoformat(),
        "observations": len(frame),
        "best_model_by_rmse": summary["best_model_by_rmse"],
        "model_metrics": summary["model_metrics"],
        "_figure": figure,
    }

def find_largest_errors(model, start_date=None, end_date=None, top_n=5):
    frame = filter_period(start_date, end_date)
    selected_models = list(MODEL_COLUMNS) if model == "All models" else [model]
    if any(name not in MODEL_COLUMNS for name in selected_models):
        raise ValueError("Unsupported model name.")
    rows = []
    for name in selected_models:
        column = MODEL_COLUMNS[name]
        errors = (frame[column] - frame["actual_mw"]).abs()
        for index in errors.nlargest(int(np.clip(top_n, 1, 20))).index:
            row = frame.loc[index]
            rows.append({
                "model": name,
                "datetime": row["datetime"],
                "actual_mw": row["actual_mw"],
                "predicted_mw": row[column],
                "absolute_error_mw": abs(row[column] - row["actual_mw"]),
            })
    result = pd.DataFrame(rows).sort_values("absolute_error_mw", ascending=False)
    return {
        "period_start": frame["datetime"].min().isoformat(),
        "period_end": frame["datetime"].max().isoformat(),
        "largest_errors": records_for_json(result, limit=60),
    }

def compare_models(start_date=None, end_date=None, models=None):
    frame = filter_period(start_date, end_date)
    selected = models or list(MODEL_COLUMNS)
    table = metrics_for(frame, selected)
    return {
        "period_start": frame["datetime"].min().isoformat(),
        "period_end": frame["datetime"].max().isoformat(),
        "observations": len(frame),
        "ranking_by_rmse": records_for_json(table),
        "best_model_by_rmse": table.iloc[0]["model"],
    }

def analyse_demand_patterns(start_date=None, end_date=None, group_by="hour"):
    frame = filter_period(start_date, end_date)
    if group_by == "hour":
        frame["group"] = frame["datetime"].dt.hour
        x_title = "Hour of day"
    elif group_by == "day_of_week":
        ordered = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
        frame["group"] = pd.Categorical(
            frame["datetime"].dt.day_name(), categories=ordered, ordered=True
        )
        x_title = "Day of week"
    elif group_by == "month":
        frame["group"] = frame["datetime"].dt.to_period("M").astype(str)
        x_title = "Month"
    else:
        raise ValueError("group_by must be hour, day_of_week or month.")
    grouped = frame.groupby("group", observed=True)["actual_mw"].agg(
        mean_mw="mean", minimum_mw="min", maximum_mw="max", observations="size"
    ).reset_index()
    grouped["group"] = grouped["group"].astype(str)
    peak = grouped.loc[grouped["mean_mw"].idxmax()]
    figure = go.Figure(go.Bar(
        x=grouped["group"], y=grouped["mean_mw"],
        marker_color="#0072B2", name="Mean actual demand",
    ))
    figure.update_layout(
        title=f"Average actual demand by {group_by.replace('_', ' ')}",
        xaxis_title=x_title, yaxis_title="Mean demand (MW)",
        template="plotly_white", height=500,
    )
    return {
        "period_start": frame["datetime"].min().isoformat(),
        "period_end": frame["datetime"].max().isoformat(),
        "group_by": group_by,
        "highest_average_group": str(peak["group"]),
        "highest_average_mw": json_value(peak["mean_mw"]),
        "pattern_table": records_for_json(grouped, limit=40),
        "_figure": figure,
    }

print(compare_models())

{'period_start': '2020-12-18T00:30:00', 'period_end': '2021-03-18T00:00:00', 'observations': 4320, 'ranking_by_rmse': [{'model': 'LSTM', 'observations': 4320, 'mae_mw': 62.245, 'rmse_mw': 82.759, 'mape_pct': 0.839, 'r2': 0.994}, {'model': 'XGBoost', 'observations': 4320, 'mae_mw': 66.726, 'rmse_mw': 89.457, 'mape_pct': 0.906, 'r2': 0.993}, {'model': 'Persistence baseline', 'observations': 4320, 'mae_mw': 137.402, 'rmse_mw': 170.164, 'mape_pct': 1.846, 'r2': 0.975}], 'best_model_by_rmse': 'LSTM'}


## 4. OpenAI function-calling analyst

In Google Colab, add `OPENAI_API_KEY` under the key icon in the left sidebar. The key is read at runtime and is never printed or stored in this notebook.

In [4]:
from openai import OpenAI

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if IN_COLAB:
    try:
        from google.colab import userdata
        OPENAI_API_KEY = userdata.get("OPENAI_API_KEY") or OPENAI_API_KEY
    except Exception:
        pass

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.6-luna")
client = OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None
print("GenAI tool-calling mode ready." if client else "Add OPENAI_API_KEY to enable GenAI routing.")
print(f"Configured model: {OPENAI_MODEL}")

NULLABLE_DATE = {"type": ["string", "null"], "description": "ISO date or datetime; use null for the full available period."}
MODEL_ENUM = ["Persistence baseline", "XGBoost", "LSTM"]
TOOLS = [
    {
        "type": "function", "name": "get_observation",
        "description": "Look up actual demand, predictions and errors at an exact or nearest timestamp.",
        "parameters": {
            "type": "object", "properties": {"timestamp": {"type": "string"}},
            "required": ["timestamp"], "additionalProperties": False,
        }, "strict": True,
    },
    {
        "type": "function", "name": "analyse_period",
        "description": "Calculate verified demand statistics and all model metrics for a requested period.",
        "parameters": {
            "type": "object", "properties": {
                "start_date": {"type": "string"}, "end_date": {"type": "string"}
            }, "required": ["start_date", "end_date"], "additionalProperties": False,
        }, "strict": True,
    },
    {
        "type": "function", "name": "plot_period",
        "description": "Create a line chart of actual demand and every model prediction. Use null dates for a reproducible random consecutive-day window.",
        "parameters": {
            "type": "object", "properties": {
                "start_date": NULLABLE_DATE, "end_date": NULLABLE_DATE,
                "days": {"type": "integer", "minimum": 1, "maximum": 31},
                "seed": {"type": "integer", "minimum": 0},
            },
            "required": ["start_date", "end_date", "days", "seed"],
            "additionalProperties": False,
        }, "strict": True,
    },
    {
        "type": "function", "name": "find_largest_errors",
        "description": "Find the largest absolute forecast errors for one model or all models.",
        "parameters": {
            "type": "object", "properties": {
                "model": {"type": "string", "enum": ["All models"] + MODEL_ENUM},
                "start_date": NULLABLE_DATE, "end_date": NULLABLE_DATE,
                "top_n": {"type": "integer", "minimum": 1, "maximum": 20},
            },
            "required": ["model", "start_date", "end_date", "top_n"],
            "additionalProperties": False,
        }, "strict": True,
    },
    {
        "type": "function", "name": "compare_models",
        "description": "Compare selected forecasting models using MAE, RMSE, MAPE and R-squared.",
        "parameters": {
            "type": "object", "properties": {
                "start_date": NULLABLE_DATE, "end_date": NULLABLE_DATE,
                "models": {"type": "array", "items": {"type": "string", "enum": MODEL_ENUM}, "minItems": 1},
            },
            "required": ["start_date", "end_date", "models"],
            "additionalProperties": False,
        }, "strict": True,
    },
    {
        "type": "function", "name": "analyse_demand_patterns",
        "description": "Analyse and chart actual-demand patterns by hour, day of week or month.",
        "parameters": {
            "type": "object", "properties": {
                "start_date": NULLABLE_DATE, "end_date": NULLABLE_DATE,
                "group_by": {"type": "string", "enum": ["hour", "day_of_week", "month"]},
            },
            "required": ["start_date", "end_date", "group_by"],
            "additionalProperties": False,
        }, "strict": True,
    },
]

APPROVED_FUNCTIONS = {
    "get_observation": get_observation,
    "analyse_period": analyse_period,
    "plot_period": plot_period,
    "find_largest_errors": find_largest_errors,
    "compare_models": compare_models,
    "analyse_demand_patterns": analyse_demand_patterns,
}

SYSTEM_INSTRUCTIONS = """
You are the answering layer for a NSW electricity-demand forecasting portfolio project.
For every numerical, date-specific, comparison, error, pattern or chart request, call one or more
approved tools before answering. Never claim the data is unavailable until the relevant tool has
been tried. Never invent a value or causal explanation. Treat all results as a historical backtest,
not a live forecast. State the selected date range, metric and model when relevant. If a chart tool
was used, briefly explain what the chart shows. Keep the final answer concise and interview-friendly.
""".strip()

def recent_history_text(history, maximum_messages=6):
    lines = []
    for item in (history or [])[-maximum_messages:]:
        content = item.get("content", "") if isinstance(item, dict) else ""
        if isinstance(content, str):
            lines.append(f"{item.get('role', 'unknown')}: {content[:600]}")
    return "\n".join(lines)

def run_approved_tool(name, arguments):
    if name not in APPROVED_FUNCTIONS:
        raise ValueError(f"Tool is not approved: {name}")
    result = APPROVED_FUNCTIONS[name](**arguments)
    figure = result.pop("_figure", None) if isinstance(result, dict) else None
    return result, figure

def local_fallback(question):
    lowered = question.lower()
    if any(word in lowered for word in ["plot", "chart", "graph"]):
        result = plot_period(None, None, 5, 42)
        figure = result.pop("_figure")
        return (
            f"Generated a five-day historical chart from {result['chart_start']} to {result['chart_end']}. "
            f"{result['best_model_by_rmse']} had the lowest RMSE in this window.",
            figure,
        )
    table = compare_models()
    best = table["ranking_by_rmse"][0]
    return (
        f"Add OPENAI_API_KEY to Colab Secrets for natural-language tool routing. "
        f"Verified fallback: {best['model']} has the lowest loaded-period RMSE "
        f"at {best['rmse_mw']:.2f} MW.",
        None,
    )

def ask_forecast(message, history):
    question = str(message).strip()
    if not question:
        return "Please enter a question.", None
    if len(question) > 700:
        return "Please keep the question below 700 characters.", None
    if client is None:
        return local_fallback(question)

    user_input = f"""
Loaded data coverage: {predictions['datetime'].min().isoformat()} to {predictions['datetime'].max().isoformat()}
Loaded observations: {len(predictions)}
Recent conversation:
{recent_history_text(history)}
Current question: {question}
""".strip()
    conversation = [{"role": "user", "content": user_input}]
    latest_figure = None

    try:
        for _ in range(5):
            response = client.responses.create(
                model=OPENAI_MODEL,
                instructions=SYSTEM_INSTRUCTIONS,
                input=conversation,
                tools=TOOLS,
                tool_choice="auto",
                parallel_tool_calls=False,
                max_output_tokens=600,
            )
            calls = [item for item in response.output if item.type == "function_call"]
            if not calls:
                return response.output_text or "Analysis completed.", latest_figure
            conversation.extend(response.output)
            for call in calls:
                try:
                    arguments = json.loads(call.arguments)
                    result, figure = run_approved_tool(call.name, arguments)
                    if figure is not None:
                        latest_figure = figure
                    output = {"status": "success", "tool": call.name, "result": result}
                except Exception as error:
                    output = {"status": "error", "tool": call.name, "message": str(error)}
                conversation.append({
                    "type": "function_call_output",
                    "call_id": call.call_id,
                    "output": json.dumps(output),
                })
        return "The analysis exceeded the five-step tool limit. Please narrow the question.", latest_figure
    except Exception as error:
        return f"The GenAI request failed ({type(error).__name__}): {error}", latest_figure

GenAI tool-calling mode ready.
Configured model: gpt-5.6-luna


## 5. Launch the Gradio analyst

In [7]:
import gradio as gr

def respond(message, history):
    question = str(message).strip()
    if not question:
        return "", history, None
    answer, figure = ask_forecast(question, history)
    updated = list(history or [])
    updated.append({"role": "user", "content": question})
    updated.append({"role": "assistant", "content": answer})
    return "", updated, figure

with gr.Blocks(title="NSW Electricity Forecasting GenAI Analyst") as demo:
    gr.Markdown("# NSW Electricity Forecasting GenAI Analyst")
    gr.Markdown(
        f"Grounded in **{len(predictions):,} historical observations** from "
        f"**{predictions['datetime'].min():%d %b %Y}** to "
        f"**{predictions['datetime'].max():%d %b %Y}**. "
        "It can plot model outputs, compare models, find large errors and analyse demand patterns."
    )
    chatbot = gr.Chatbot(
        value=[{"role": "assistant", "content": "Ask me to analyse or chart the historical forecasting results."}],
        height=430,
    )
    chart = gr.Plot(label="Generated chart")
    with gr.Row():
        prompt = gr.Textbox(
            placeholder="Example: Plot all models for any five random consecutive days",
            label="Question",
            scale=8,
        )
        send = gr.Button("Send", variant="primary", scale=1)
        clear = gr.Button("Clear", scale=1)
    gr.Examples(
        examples=[
            "Plot actual demand and all models for any five random consecutive days.",
            "Compare LSTM, XGBoost and the persistence baseline over the full loaded period.",
            "Show the five largest LSTM forecast errors.",
            "Analyse average demand by hour of day and chart it.",
            "What happened at the timestamp with the largest XGBoost error?",
        ],
        inputs=prompt,
    )
    send.click(respond, [prompt, chatbot], [prompt, chatbot, chart])
    prompt.submit(respond, [prompt, chatbot], [prompt, chatbot, chart])
    clear.click(lambda: ("", [], None), None, [prompt, chatbot, chart])

demo.launch(share=IN_COLAB)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7165811cb5f8c17dd4.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
